In [7]:
import subprocess
subprocess.run(["pip", "install", "streamlit", "plotly", "openpyxl"], check=True)
print("Dependencias instaladas")

Dependencias instaladas


In [15]:
codigo = """
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os, pickle

st.set_page_config(page_title="Mn EA - Random Forest", page_icon="RF", layout="wide")

st.markdown(\"\"\"
<style>
@import url("https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&family=JetBrains+Mono:wght@400;600&display=swap");
html, body, [class*="css"] { font-family: "Inter", sans-serif; }
.stApp { background-color: #0f1117; }
.header-block {
    background: linear-gradient(135deg,#1a1f2e 0%,#0f1117 100%);
    border-left: 4px solid #00c896;
    padding: 24px 32px; border-radius: 8px; margin-bottom: 28px;
}
.header-block h1 { color:#f0f4f8; font-size:1.8rem; font-weight:700; margin:0 0 4px 0; }
.header-block p  { color:#8892a4; font-size:0.9rem; margin:0; }
.kpi-card { background:#1a1f2e; border:1px solid #2a3044; border-radius:10px; padding:20px 24px; text-align:center; }
.kpi-label { color:#8892a4; font-size:0.72rem; font-weight:600; text-transform:uppercase; letter-spacing:1px; margin-bottom:6px; }
.kpi-value { color:#00c896; font-family:"JetBrains Mono",monospace; font-size:1.6rem; font-weight:700; }
.kpi-unit  { color:#8892a4; font-size:0.78rem; margin-top:2px; }
.pred-box  { background:#1a1f2e; border:2px solid #00c896; border-radius:12px; padding:28px 32px; text-align:center; margin:16px 0; }
.pred-value { color:#00c896; font-family:"JetBrains Mono",monospace; font-size:2.6rem; font-weight:700; }
.pred-label { color:#8892a4; font-size:0.85rem; margin-top:6px; }
.badge-ok   { background:#0d3326; color:#00c896; border:1px solid #00c896; border-radius:20px; padding:5px 16px; font-size:0.8rem; font-weight:600; display:inline-block; margin-top:10px; }
.badge-warn { background:#3a1a1a; color:#ff6b6b; border:1px solid #ff6b6b; border-radius:20px; padding:5px 16px; font-size:0.8rem; font-weight:600; display:inline-block; margin-top:10px; }
.section-title { color:#f0f4f8; font-size:1rem; font-weight:600; border-bottom:1px solid #2a3044; padding-bottom:8px; margin-bottom:16px; }
div[data-testid="stSelectbox"] label,
div[data-testid="stSlider"] label,
div[data-testid="stNumberInput"] label { color:#8892a4 !important; font-size:0.78rem; font-weight:600; text-transform:uppercase; letter-spacing:0.8px; }
.footnote { color:#4a5568; font-size:0.75rem; text-align:center; margin-top:32px; }
</style>
\"\"\", unsafe_allow_html=True)

st.markdown(\"\"\"
<div class="header-block">
    <h1>Prediccion de Mn EA - Random Forest</h1>
    <p>Modelo predictivo - Electroobtencion de Zinc - CJM ELE</p>
</div>
\"\"\", unsafe_allow_html=True)

RANGO_MN_SOL_PURA = (5, 8)
RANGO_MN_EA       = (12, 15)
ARCHIVO_EXCEL     = "Predicciones_Mn_EA_RandomForest.xlsx"
ARCHIVO_EXCEL_ORIG = "Datos reales CJM ELE.xlsx"
ARCHIVO_MODELO    = "modelo_rf_mn.pkl"
ARCHIVO_SCALER    = "scaler_rf_mn.pkl"

if not os.path.exists(ARCHIVO_MODELO):
    st.error("No se encontro el modelo entrenado. Ejecuta primero el notebook Random Forest (celda 5) y luego la celda 14.")
    st.stop()

modelo = pickle.load(open(ARCHIVO_MODELO, "rb"))
scaler = pickle.load(open(ARCHIVO_SCALER, "rb"))

features_display = ["T de EA (C)", "Zn Sol Pura (g/L)", "Mn Sol Pura",
                    "Fe (mg/L)", "Sb (mg/L)", "Cu (mg/L)",
                    "Densidad (g/L)", "Acidez (%)", "Horas de deposito", "Peso Deposito (kg)"]

features_excel = ["T de EA (\u00b0C)", "Zn Sol Pura (g/L)", "Mn Sol Pura",
                  "Fe (mg/L)", "Sb (mg/L)", "Cu (mg/L)",
                  "Densidad (g/L)", "Acidez (%)", "Horas de dep\u00f3sito", "Peso Dep\u00f3sito (kg)"]

# Variables excluidas del grafico de correlacion (para ver relacion con las demas)
EXCLUIR_CORR = ["Mn Sol Pura", "Zn Sol Pura (g/L)"]

if os.path.exists(ARCHIVO_EXCEL):
    df = pd.read_excel(ARCHIVO_EXCEL)
    mae    = df["Error Mn EA (g/L)"].mean()
    r2     = 1 - ((df["Mn EA Real (g/L)"] - df["Mn EA Predicho (g/L)"])**2).sum() / \
                 ((df["Mn EA Real (g/L)"] - df["Mn EA Real (g/L)"].mean())**2).sum()
    n      = len(df)
    pct_ok = df["Mn EA Predicho dentro rango (12-15)"].mean() * 100
else:
    df = None
    mae, r2, n, pct_ok = None, None, None, None

# Cargar dataset original para Zn EA real
if os.path.exists(ARCHIVO_EXCEL_ORIG):
    df_orig = pd.read_excel(ARCHIVO_EXCEL_ORIG, sheet_name="Hoja1")
else:
    df_orig = None

# ── KPIs ──────────────────────────────────────────────────────────────
if df is not None:
    c1, c2, c3, c4, c5 = st.columns(5)
    kpis = [
        (c1, "MAE Mn EA",            f"{mae:.3f}",    "g/L"),
        (c2, "R2 Mn EA",             f"{r2:.4f}",     "coef."),
        (c3, "Registros prueba",      str(n),          "muestras"),
        (c4, "Mn EA en rango 12-15",  f"{pct_ok:.1f}%","predicciones"),
        (c5, "Rango Mn Sol Pura",     "5 a 8 g/L",    "referencia"),
    ]
    for col, label, value, unit in kpis:
        with col:
            st.markdown(
                '<div class="kpi-card">'
                '<div class="kpi-label">' + label + '</div>'
                '<div class="kpi-value">' + value + '</div>'
                '<div class="kpi-unit">'  + unit  + '</div>'
                '</div>',
                unsafe_allow_html=True
            )
    st.markdown("<br>", unsafe_allow_html=True)

# ── SECCION 1: PREDICCION ─────────────────────────────────────────────
st.markdown("<p class='section-title'>Ingresar Parametros para Nueva Prediccion</p>", unsafe_allow_html=True)

with st.form("form_prediccion"):
    col1, col2, col3 = st.columns(3)
    with col1:
        t_ea        = st.number_input("T de EA (C)",           min_value=20.0,   max_value=60.0,   value=37.5,   step=0.1)
        mn_sol_pura = st.number_input("Mn Sol Pura (g/L)",     min_value=3.0,    max_value=12.0,   value=6.5,    step=0.01)
        fe          = st.number_input("Fe (mg/L)",              min_value=0.0,    max_value=20.0,   value=2.5,    step=0.01)
        densidad    = st.number_input("Densidad (g/L)",         min_value=1250.0, max_value=1350.0, value=1302.0, step=0.1)
    with col2:
        zn_sol_pura = st.number_input("Zn Sol Pura (g/L)",     min_value=100.0,  max_value=220.0,  value=162.0,  step=0.1)
        sb          = st.number_input("Sb (mg/L)",              min_value=0.0,    max_value=0.1,    value=0.012,  step=0.001, format="%.3f")
        cu          = st.number_input("Cu (mg/L)",              min_value=0.0,    max_value=1.0,    value=0.14,   step=0.001, format="%.3f")
        acidez      = st.number_input("Acidez (%)",             min_value=100.0,  max_value=250.0,  value=188.0,  step=0.1)
    with col3:
        horas       = st.number_input("Horas de deposito",      min_value=1.0,    max_value=100.0,  value=44.0,   step=0.5)
        peso        = st.number_input("Peso Deposito (kg)",     min_value=10.0,   max_value=150.0,  value=73.0,   step=0.1)
        st.markdown("<br><br>", unsafe_allow_html=True)
        submitted   = st.form_submit_button("Calcular Prediccion", use_container_width=True)

if submitted:
    vals    = [[t_ea, zn_sol_pura, mn_sol_pura, fe, sb, cu, densidad, acidez, horas, peso]]
    X_new   = pd.DataFrame(vals, columns=features_excel)
    X_sc    = scaler.transform(X_new)
    mn_pred = float(modelo.predict(X_sc)[0])

    ok_sol = RANGO_MN_SOL_PURA[0] <= mn_sol_pura <= RANGO_MN_SOL_PURA[1]
    ok_ea  = RANGO_MN_EA[0] <= mn_pred <= RANGO_MN_EA[1]

    badge_sol = '<span class="badge-ok">Mn Sol Pura dentro de 5 a 8 g/L</span>' if ok_sol else '<span class="badge-warn">Mn Sol Pura fuera de 5 a 8 g/L</span>'
    badge_ea  = '<span class="badge-ok">Mn EA dentro de 12 a 15 g/L</span>'     if ok_ea  else '<span class="badge-warn">Mn EA fuera de 12 a 15 g/L</span>'

    rc1, rc2, rc3 = st.columns([1, 2, 1])
    with rc2:
        st.markdown(
            '<div class="pred-box">'
            '<div class="pred-label">Mn EA Predicho (Random Forest)</div>'
            '<div class="pred-value">' + f"{mn_pred:.3f}" + ' <span style="font-size:1.2rem">g/L</span></div>'
            '<div>' + badge_sol + ' &nbsp; ' + badge_ea + '</div>'
            '</div>',
            unsafe_allow_html=True
        )
    with st.expander("Resumen de los parametros ingresados"):
        resumen = pd.DataFrame({"Parametro": features_display, "Valor ingresado": vals[0]})
        st.dataframe(resumen, use_container_width=True, hide_index=True)

# ── SECCION 2: GRAFICO PREDICHO vs REAL ──────────────────────────────
if df is not None:
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("<p class='section-title'>Predicciones vs. Valores Reales (conjunto de prueba)</p>", unsafe_allow_html=True)

    x_min = df["Mn EA Real (g/L)"].min() - 0.3
    x_max = df["Mn EA Real (g/L)"].max() + 0.3

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["Mn EA Real (g/L)"], y=df["Mn EA Predicho (g/L)"],
        mode="markers",
        marker=dict(color="#ff6b6b", size=7, opacity=0.6, line=dict(width=0.5, color="#fff")),
        name="Predicciones",
        hovertemplate="Real: %{x:.3f} g/L<br>Predicho: %{y:.3f} g/L<extra></extra>"
    ))
    fig.add_trace(go.Scatter(
        x=[x_min, x_max], y=[x_min, x_max],
        mode="lines", line=dict(color="#00c896", width=2, dash="dash"), name="Linea ideal"
    ))
    fig.add_hrect(
        y0=RANGO_MN_EA[0], y1=RANGO_MN_EA[1],
        fillcolor="#00c896", opacity=0.06, line_width=0,
        annotation_text="Rango objetivo 12 a 15 g/L",
        annotation_position="top left",
        annotation_font=dict(color="#00c896", size=11)
    )
    fig.update_layout(
        paper_bgcolor="#1a1f2e", plot_bgcolor="#1a1f2e",
        font=dict(family="Inter", color="#8892a4", size=12),
        title=dict(text="Mn EA - Predicho vs. Real (Random Forest)", font=dict(color="#f0f4f8", size=15), x=0.01),
        xaxis=dict(title="Mn EA Real (g/L)", gridcolor="#2a3044", color="#8892a4"),
        yaxis=dict(title="Mn EA Predicho (g/L)", gridcolor="#2a3044", color="#8892a4"),
        legend=dict(bgcolor="#0f1117", bordercolor="#2a3044", borderwidth=1),
        margin=dict(l=40, r=20, t=50, b=40), height=420,
    )
    st.plotly_chart(fig, use_container_width=True)

    # ── SECCION 3: CORRELACIONES Mn EA y Zn EA (sin Mn Sol Pura ni Zn Sol Pura) ──
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("<p class='section-title'>Correlacion de Variables con Mn EA Predicho y Zn EA Real</p>", unsafe_allow_html=True)

    # Variables a mostrar (excluyendo Mn Sol Pura y Zn Sol Pura)
    vars_corr = [v for v in features_excel if v not in EXCLUIR_CORR]
    vars_corr_display = [v for v in features_display if v not in ["Mn Sol Pura", "Zn Sol Pura (g/L)"]]

    # Correlacion con Mn EA Predicho
    corr_mn = df[vars_corr + ["Mn EA Predicho (g/L)"]].corr()["Mn EA Predicho (g/L)"].drop("Mn EA Predicho (g/L)")

    # Correlacion con Zn EA Real (desde dataset original, alineado con X_test)
    if df_orig is not None:
        df_test_idx = df.index
        df_zn = df_orig.copy().reset_index(drop=True)
        # Usamos los mismos indices del conjunto de prueba si estan disponibles
        # Si no, tomamos una muestra aleatoria del mismo tamanio para calcular correlacion
        if len(df_zn) >= len(df):
            df_zn_test = df_zn.sample(n=len(df), random_state=42).reset_index(drop=True)
        else:
            df_zn_test = df_zn.reset_index(drop=True)

        corr_zn = df_zn_test[vars_corr + ["Zn EA (g/L)"]].corr()["Zn EA (g/L)"].drop("Zn EA (g/L)")
        zn_disponible = True
    else:
        zn_disponible = False

    # Grafico de barras dobles
    fig_corr = go.Figure()
    fig_corr.add_trace(go.Bar(
        name="Correlacion con Mn EA Predicho",
        x=vars_corr_display,
        y=corr_mn.values,
        marker_color="#ff6b6b",
        opacity=0.85
    ))
    if zn_disponible:
        fig_corr.add_trace(go.Bar(
            name="Correlacion con Zn EA Real",
            x=vars_corr_display,
            y=corr_zn.values,
            marker_color="#4a9eff",
            opacity=0.85
        ))
    fig_corr.add_hline(y=0, line_color="white", line_width=0.8)
    fig_corr.update_layout(
        barmode="group",
        paper_bgcolor="#1a1f2e", plot_bgcolor="#1a1f2e",
        font=dict(family="Inter", color="#8892a4", size=12),
        title=dict(text="Correlacion de Variables con Mn EA Predicho y Zn EA Real (excluye Mn y Zn Sol Pura)",
                   font=dict(color="#f0f4f8", size=13), x=0.01),
        xaxis=dict(title="Variable", gridcolor="#2a3044", color="#8892a4", tickangle=-25),
        yaxis=dict(title="Coeficiente de correlacion", gridcolor="#2a3044", color="#8892a4"),
        legend=dict(bgcolor="#0f1117", bordercolor="#2a3044", borderwidth=1),
        margin=dict(l=40, r=20, t=60, b=80), height=440,
    )
    st.plotly_chart(fig_corr, use_container_width=True)

    # ── SECCION 4: TABLA RELACIONAL ───────────────────────────────────
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("<p class='section-title'>Analisis Relacional - Variables mas importantes para Mn EA y Zn EA</p>", unsafe_allow_html=True)

    def clasificar(r):
        if r >= 0.5:   return "Fuerte directa"
        elif r >= 0.2: return "Moderada directa"
        elif r > -0.2: return "Sin relacion clara"
        elif r > -0.5: return "Moderada inversa"
        else:          return "Fuerte inversa"

    def comportamiento(r_mn, r_zn):
        ambas_pos = r_mn > 0.05 and r_zn > 0.05
        ambas_neg = r_mn < -0.05 and r_zn < -0.05
        solo_mn   = abs(r_mn) > 0.05 and abs(r_zn) <= 0.05
        solo_zn   = abs(r_mn) <= 0.05 and abs(r_zn) > 0.05
        opuestas  = (r_mn > 0.05 and r_zn < -0.05) or (r_mn < -0.05 and r_zn > 0.05)
        if ambas_pos:  return "Ambas suben juntas"
        elif ambas_neg: return "Ambas bajan juntas"
        elif solo_mn:  return "Solo afecta a Mn EA"
        elif solo_zn:  return "Solo afecta a Zn EA"
        elif opuestas: return "Direcciones opuestas"
        else:          return "Sin relacion en ninguna"

    tabla_rel = []
    for var_e, var_d in zip(vars_corr, vars_corr_display):
        r_mn = round(float(corr_mn.get(var_e, 0)), 4)
        r_zn = round(float(corr_zn.get(var_e, 0)), 4) if zn_disponible else None
        fila = {
            "Variable": var_d,
            "Corr. Mn EA": r_mn,
            "Relacion Mn EA": clasificar(r_mn),
            "Corr. Zn EA": r_zn if r_zn is not None else "N/D",
            "Relacion Zn EA": clasificar(r_zn) if r_zn is not None else "N/D",
            "Comportamiento conjunto": comportamiento(r_mn, r_zn) if r_zn is not None else "N/D",
        }
        tabla_rel.append(fila)

    df_rel = pd.DataFrame(tabla_rel).sort_values("Corr. Mn EA", ascending=False).reset_index(drop=True)
    st.dataframe(df_rel, use_container_width=True, hide_index=True)

st.markdown(
    '<p class="footnote">Modelo: Random Forest - 400 arboles - max depth 8 - Division 80/20 - Datos: CJM ELE</p>',
    unsafe_allow_html=True
)
"""

with open("app_rf_mn.py", "w", encoding="utf-8") as f:
    f.write(codigo)
print("app_rf_mn.py creado correctamente")

app_rf_mn.py creado correctamente


In [16]:
import subprocess, threading, time

def run():
    subprocess.run(["streamlit", "run", "app_rf_mn.py",
                    "--server.port=8501", "--server.headless=true"])

threading.Thread(target=run, daemon=True).start()
time.sleep(3)
print("App corriendo en: http://localhost:8501")

App corriendo en: http://localhost:8501
